# Lab 4: Observability & Monitoring for AI Agents

## Why this matters
The course uses LangSmith for tracing (Week 4). But monitoring agents in production needs more:
- **Structured logging** — machine-readable, searchable, correlated by session
- **Metrics** — latency, token usage, error rates, cache hit rates
- **Tracing** — full request flow across tools and models
- **Alerting** — know when your agent is failing before your users tell you

## What we'll build
1. **Structured logging** with correlation IDs
2. **Metrics collection** (latency, tokens, errors)
3. **Custom LangGraph tracing** — see the full agent trajectory
4. **Cost dashboarding** — track spend per session/user
5. **Alerting patterns** — detect agent anomalies automatically

---

In [ ]:
import os
import json
import time
import uuid
import logging
from typing import Optional, Any
from dataclasses import dataclass, field, asdict
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
print("Setup complete.")

---
## Part 1: Structured Logging

**The problem with print():** It's not searchable, not parseable, and can't be correlated.

**Structured logging** outputs JSON — every log entry is filterable, queryable, and correlated by request ID.

In [ ]:
import logging
import sys

# NOTE ON PRODUCTION LOG SHIPPING:
# AgentLogger below writes JSON to stdout — the correct 12-factor-app pattern.
# In production, your infrastructure captures stdout and ships it:
#
#   AWS CloudWatch:    stdout → CloudWatch Logs (automatic with ECS/Lambda)
#   GCP Cloud Logging: stdout → Cloud Logging (automatic with Cloud Run)
#   Grafana Loki:      pip install python-logging-loki → LokiHandler
#   Datadog:           pip install datadog → DatadogHandler
#   Local dev:         stdout (JSON) → grep / jq for filtering
#
# Example Loki handler (drop-in replacement for stdout):
#   import logging_loki
#   handler = logging_loki.LokiHandler(
#       url="http://loki:3100/loki/api/v1/push",
#       tags={"service": "agent-service"},
#       auth=("user", "password"),
#   )
#
# Key rule: always write JSON to stdout first; handler routing is infra config.

class AgentLogger:
    """Structured logger that outputs JSON lines — works without extra packages."""
    
    def __init__(self, service_name: str):
        self.service = service_name
    
    def _emit(self, level: str, message: str, **kwargs):
        entry = {
            "ts": datetime.utcnow().isoformat() + "Z",
            "level": level,
            "service": self.service,
            "message": message,
            **kwargs
        }
        print(json.dumps(entry))
    
    def info(self, msg: str, **kw):  self._emit("INFO", msg, **kw)
    def warn(self, msg: str, **kw):  self._emit("WARN", msg, **kw)
    def error(self, msg: str, **kw): self._emit("ERROR", msg, **kw)
    def debug(self, msg: str, **kw): self._emit("DEBUG", msg, **kw)

log = AgentLogger("agent-service")

# Demo: correlated logs across a request lifecycle
request_id = str(uuid.uuid4())[:8]
session_id = "sess-abc123"

log.info("request_started", request_id=request_id, session_id=session_id, user="user-42")
log.info("llm_call_started", request_id=request_id, model="gpt-4o-mini", prompt_tokens=250)
log.info("llm_call_completed", request_id=request_id, model="gpt-4o-mini", completion_tokens=120, latency_ms=843)
log.info("request_completed", request_id=request_id, total_latency_ms=901, status="success")

---
## Part 2: Metrics Collection

Metrics answer **aggregate questions** over time:
- P95 latency is 4.2s — is that trending up or down?
- Error rate spiked to 5% at 14:00 — what changed?
- Token usage is 3x this week — which users/sessions are driving that?

In [ ]:
from collections import defaultdict, deque
import statistics

# NOTE ON PRODUCTION METRICS:
# MetricsCollector below uses in-memory Python lists — great for learning the concepts,
# but incompatible with Prometheus scraping. Prometheus works by:
#   1. You expose a /metrics HTTP endpoint (via prometheus_client library)
#   2. Prometheus scrapes it on a schedule (pull model)
#   3. You use Counter/Histogram/Gauge types, NOT plain lists
#
# Production path:
#   pip install prometheus-client
#   from prometheus_client import Counter, Histogram, start_http_server
#   REQUEST_LATENCY = Histogram("agent_latency_seconds", "Latency", buckets=[.1,.5,1,2,5,10])
#   REQUEST_LATENCY.observe(latency_ms / 1000)
#   start_http_server(9090)  # Prometheus scrapes :9090/metrics
#
# This notebook teaches the concepts; swap the storage backend for prod.

@dataclass
class MetricsCollector:
    """In-memory metrics. In prod: push to Prometheus, Datadog, or CloudWatch."""
    
    # Bounded deque prevents unbounded memory growth in long-running services.
    # 10,000 samples ≈ ~300KB. Older samples are discarded automatically.
    MAX_LATENCY_SAMPLES: int = 10_000
    
    token_counts: dict = field(default_factory=lambda: defaultdict(int))
    error_counts: dict = field(default_factory=lambda: defaultdict(int))
    call_counts: dict  = field(default_factory=lambda: defaultdict(int))
    costs_usd: list    = field(default_factory=list)
    
    # deque with maxlen automatically evicts oldest entries — no manual pruning needed
    latencies_ms: deque = field(default_factory=lambda: deque(maxlen=10_000))
    
    def record_call(self, model: str, input_tokens: int, output_tokens: int, 
                    latency_ms: float, success: bool = True):
        self.latencies_ms.append(latency_ms)
        self.token_counts[f"{model}.input"]  += input_tokens
        self.token_counts[f"{model}.output"] += output_tokens
        self.call_counts[model] += 1
        if not success:
            self.error_counts[model] += 1
        
        INPUT_COST  = {"gpt-4o": 2.50, "gpt-4o-mini": 0.15}
        OUTPUT_COST = {"gpt-4o": 10.0, "gpt-4o-mini": 0.60}
        cost = (input_tokens  / 1e6 * INPUT_COST.get(model, 0.15) +
                output_tokens / 1e6 * OUTPUT_COST.get(model, 0.60))
        self.costs_usd.append(cost)
    
    def report(self) -> dict:
        lats = list(self.latencies_ms) or [0]
        sorted_lats = sorted(lats)
        n = len(sorted_lats)
        
        def percentile(p):
            idx = int(n * p / 100)
            return sorted_lats[min(idx, n-1)]
        
        total_calls = sum(self.call_counts.values())
        total_errors = sum(self.error_counts.values())
        
        return {
            "latency": {
                "p50_ms":  round(percentile(50), 1),
                "p95_ms":  round(percentile(95), 1),
                "p99_ms":  round(percentile(99), 1),
                "avg_ms":  round(statistics.mean(lats), 1),
                "samples": n,
            },
            "calls": dict(self.call_counts),
            "errors": dict(self.error_counts),
            "error_rate": round(total_errors / total_calls, 4) if total_calls else 0,
            "tokens": dict(self.token_counts),
            "total_cost_usd": round(sum(self.costs_usd), 6),
            "avg_cost_per_call_usd": round(sum(self.costs_usd) / max(total_calls, 1), 6),
        }

# Simulate agent traffic
import random
metrics = MetricsCollector()

for i in range(50):
    model = random.choice(["gpt-4o-mini"] * 8 + ["gpt-4o"] * 2)
    latency = random.gauss(800, 200) if model == "gpt-4o-mini" else random.gauss(2000, 400)
    success = random.random() > 0.03
    metrics.record_call(
        model=model,
        input_tokens=random.randint(100, 800),
        output_tokens=random.randint(50, 300),
        latency_ms=max(100, latency),
        success=success
    )

report = metrics.report()
print(json.dumps(report, indent=2))

---
## Part 3: Agent Tracing — See the Full Picture

A trace captures the complete lifecycle of one agent request — all tool calls, reasoning steps, and nested LLM calls — as a tree.

In [ ]:
from contextlib import contextmanager
from contextvars import ContextVar  # async-safe context storage

@dataclass
class Span:
    span_id: str
    parent_id: Optional[str]
    name: str
    start_time: float
    end_time: Optional[float] = None
    metadata: dict = field(default_factory=dict)
    error: Optional[str] = None
    
    @property
    def duration_ms(self) -> float:
        if self.end_time:
            return (self.end_time - self.start_time) * 1000
        return 0

class Tracer:
    """Simple distributed tracer. In prod: use OpenTelemetry → Jaeger/Zipkin/Datadog.
    
    Uses ContextVar for _current_span_id so concurrent async coroutines each
    maintain their own span stack without interfering with each other.
    """
    
    def __init__(self):
        self.spans: list[Span] = []
        # ContextVar: each asyncio task/coroutine gets its own copy of this value
        self._current_span_id: ContextVar[Optional[str]] = ContextVar(
            "_current_span_id", default=None
        )
    
    @contextmanager
    def start_span(self, name: str, **metadata):
        span = Span(
            span_id=str(uuid.uuid4())[:8],
            parent_id=self._current_span_id.get(),
            name=name,
            start_time=time.time(),
            metadata=metadata
        )
        self.spans.append(span)
        # Save the previous value so we can restore it (correct nesting)
        token = self._current_span_id.set(span.span_id)
        try:
            yield span
            span.end_time = time.time()
        except Exception as e:
            span.error = str(e)
            span.end_time = time.time()
            raise
        finally:
            # Restore previous span context — works correctly even in async code
            self._current_span_id.reset(token)
    
    def print_trace(self):
        """Print the trace as a tree."""
        def print_span(span: Span, depth: int = 0):
            indent = "  " * depth
            status = "❌" if span.error else "✓"
            print(f"{indent}{status} [{span.span_id}] {span.name} ({span.duration_ms:.0f}ms)")
            if span.metadata:
                for k, v in span.metadata.items():
                    print(f"{indent}    {k}: {v}")
            if span.error:
                print(f"{indent}    error: {span.error}")
            for s in self.spans:
                if s.parent_id == span.span_id:
                    print_span(s, depth + 1)
        
        roots = [s for s in self.spans if s.parent_id is None]
        for root in roots:
            print_span(root)

# Simulate a traced agent run
tracer = Tracer()

with tracer.start_span("agent_request", user="user-42", session="sess-abc") as root:
    time.sleep(0.01)
    
    with tracer.start_span("llm_call", model="gpt-4o-mini", prompt_tokens=200):
        time.sleep(0.05)
    
    with tracer.start_span("tool_call", tool="web_search", query="LangGraph tutorial"):
        time.sleep(0.02)
        with tracer.start_span("http_request", url="https://search-api.example.com", method="GET"):
            time.sleep(0.03)
    
    with tracer.start_span("llm_call", model="gpt-4o-mini", prompt_tokens=800, note="synthesis"):
        time.sleep(0.08)

print("\nTrace tree:")
tracer.print_trace()

---
## Part 4: Anomaly Detection & Alerting

How to know when your agent is misbehaving **before** users complain.

In [ ]:
@dataclass
class AlertRule:
    name: str
    condition: str   # human-readable
    threshold: float
    window_minutes: int = 5

ALERT_RULES = [
    AlertRule("high_error_rate",    "error_rate > 5%",     threshold=0.05),
    AlertRule("high_latency_p95",   "p95 latency > 10s",   threshold=10_000),
    AlertRule("high_cost_per_call", "avg cost > $0.01",    threshold=0.01),
    AlertRule("tool_loop_detected", "tool calls > 20/req", threshold=20),
]

class AlertManager:
    def __init__(self, rules: list[AlertRule]):
        self.rules = rules
        self.fired_alerts: list[dict] = []
        # Windowed call history: list of (timestamp, success, latency_ms, cost)
        self._call_history: list[tuple[float, bool, float, float]] = []
    
    def record_call(self, success: bool, latency_ms: float, cost_usd: float):
        """Record a call for windowed alerting. Call this on every agent call."""
        self._call_history.append((time.time(), success, latency_ms, cost_usd))
    
    def _windowed_metrics(self, window_minutes: int) -> dict:
        """Compute metrics over the last N minutes only."""
        cutoff = time.time() - (window_minutes * 60)
        recent = [(s, l, c) for ts, s, l, c in self._call_history if ts >= cutoff]
        if not recent:
            return {"error_rate": 0, "p95_latency_ms": 0, "avg_cost": 0, "call_count": 0}
        
        successes = [s for s, _, _ in recent]
        latencies = sorted(l for _, l, _ in recent)
        costs = [c for _, _, c in recent]
        n = len(latencies)
        
        return {
            "error_rate": 1 - (sum(successes) / n),
            "p95_latency_ms": latencies[int(n * 0.95)] if n else 0,
            "avg_cost": sum(costs) / n,
            "call_count": n,
        }
    
    def evaluate(self, metrics: dict):
        """Check rules — point-in-time for snapshot metrics, windowed for rates."""
        for rule in self.rules:
            fired = False
            actual_value = None
            
            if rule.name == "high_error_rate":
                # Use windowed metric if we have call history, otherwise fall back to snapshot
                if self._call_history:
                    wm = self._windowed_metrics(rule.window_minutes)
                    actual_value = wm["error_rate"]
                else:
                    actual_value = metrics.get("error_rate", 0)
                fired = actual_value > rule.threshold
            
            elif rule.name == "high_latency_p95":
                if self._call_history:
                    wm = self._windowed_metrics(rule.window_minutes)
                    actual_value = wm["p95_latency_ms"]
                else:
                    actual_value = metrics.get("latency", {}).get("p95_ms", 0)
                fired = actual_value > rule.threshold
            
            elif rule.name == "high_cost_per_call":
                actual_value = metrics.get("avg_cost_per_call_usd", 0)
                fired = actual_value > rule.threshold
            
            if fired and actual_value is not None:
                alert = {
                    "ts": datetime.utcnow().isoformat(),
                    "rule": rule.name,
                    "condition": rule.condition,
                    "actual": actual_value,
                    "threshold": rule.threshold,
                    "window_minutes": rule.window_minutes,
                    "severity": "WARNING"
                }
                self.fired_alerts.append(alert)
                self._notify(alert)
    
    def _notify(self, alert: dict):
        """In prod: send to PagerDuty, Slack, or email."""
        print(f"🚨 ALERT: {alert['rule']} — {alert['condition']} "
              f"(actual: {alert['actual']:.4f}, window: {alert['window_minutes']}min)")

alert_manager = AlertManager(ALERT_RULES)

# Feed some recent call history so windowed alerting works
import random
for _ in range(20):
    alert_manager.record_call(
        success=random.random() > 0.03,
        latency_ms=random.gauss(800, 200),
        cost_usd=random.uniform(0.0001, 0.005)
    )

current_metrics = metrics.report()
alert_manager.evaluate(current_metrics)

if not alert_manager.fired_alerts:
    print("✅ All metrics within thresholds.")

---
## Part 5: LangSmith Integration (the course way, made production-ready)

In [ ]:
# LangSmith is the fastest path to production tracing for LangChain/LangGraph.
# Set these env vars before importing langchain:

LANGSMITH_SETUP = """
# Add to your .env file:
LANGCHAIN_TRACING_V2=true
LANGCHAIN_ENDPOINT=https://api.smith.langchain.com
LANGCHAIN_API_KEY=<your-key>
LANGCHAIN_PROJECT=my-agent-prod

# Then every LangChain/LangGraph call is automatically traced.
# No code changes needed.
"""

print(LANGSMITH_SETUP)

# --- Live LangSmith usage example ---
# This shows the actual code pattern — not just config.
# Requires: LANGCHAIN_API_KEY in your .env and pip install langchain-openai langsmith

LANGSMITH_LIVE_EXAMPLE = '''
# Run this block with LANGCHAIN_TRACING_V2=true set — it creates a real trace
# you can view at smith.langchain.com

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

# This call is automatically traced — appears in LangSmith as a "ChatOpenAI" span
response = llm.invoke(
    [HumanMessage(content="What is LangGraph?")],
    config={
        "run_name": "langgraph_explainer",       # name shown in LangSmith UI
        "metadata": {
            "user_id": "user-42",                # filter by user in LangSmith
            "session_id": "sess-abc",
            "version": "v2.1",
            "environment": "production",
        },
        "tags": ["prod", "rag-pipeline"],         # searchable tags
    }
)
print(response.content)
print()
print("View this trace at: https://smith.langchain.com → your project → Traces")
'''

print("LangSmith live usage pattern:")
print(LANGSMITH_LIVE_EXAMPLE)

# --- What you see in LangSmith ---
print("""After running the block above with tracing enabled, LangSmith shows:
  - Full request/response payloads
  - Token counts and cost per call  
  - Latency breakdown per step
  - user_id, session_id, version metadata (searchable/filterable)
  - Tags for grouping traces by pipeline or environment
  
  Filter by metadata in LangSmith UI:
    Tags → 'prod'       to see only production traces
    Metadata → user_id  to debug a specific user's session
""")

---
## Part 6: The Observability Dashboard (Text-Based)

In production, you'd view this in Grafana/Datadog. Here's how to build a simple terminal dashboard.

In [ ]:
def print_dashboard(metrics_report: dict):
    """Print a terminal metrics dashboard."""
    lat = metrics_report["latency"]
    
    print("\n" + "═" * 60)
    print("  AGENT OBSERVABILITY DASHBOARD")
    print("═" * 60)
    
    # Latency
    print("\n📊 LATENCY")
    print(f"  P50:  {lat['p50_ms']:>7.0f}ms")
    print(f"  P95:  {lat['p95_ms']:>7.0f}ms")
    print(f"  P99:  {lat['p99_ms']:>7.0f}ms")
    print(f"  Avg:  {lat['avg_ms']:>7.0f}ms")
    
    # Reliability
    error_rate = metrics_report["error_rate"]
    indicator = "🟢" if error_rate < 0.01 else ("🟡" if error_rate < 0.05 else "🔴")
    print(f"\n{indicator} RELIABILITY")
    print(f"  Error rate: {error_rate*100:.2f}%")
    
    # Cost
    print(f"\n💰 COST")
    print(f"  Total:         ${metrics_report['total_cost_usd']:.6f}")
    print(f"  Per call avg:  ${metrics_report['avg_cost_per_call_usd']:.6f}")
    
    # Models
    print("\n🤖 CALLS BY MODEL")
    for model, count in metrics_report["calls"].items():
        input_tok  = metrics_report["tokens"].get(f"{model}.input", 0)
        output_tok = metrics_report["tokens"].get(f"{model}.output", 0)
        print(f"  {model:20s}: {count:4d} calls | {input_tok:,} in / {output_tok:,} out tokens")
    
    print("\n" + "═" * 60)

print_dashboard(metrics.report())

---
## Part 7: Wiring Lab 3 + Lab 4 Together

Lab 3 built a FastAPI agent service. Lab 4 built observability tools. Here's how to connect them — adding structured logging, metrics, tracing, and cache-hit tracking to the Lab 3 service.

In [ ]:
# This shows the integration pattern — add this middleware to the Lab 3 agent_service.py.
# The key addition is recording metrics and cache-hit status on every /chat request.

LAB3_LAB4_INTEGRATION = '''
# --- Add to agent_service.py (Lab 3) to get Lab 4 observability ---
# Place this at the top of agent_service.py after existing imports:

from collections import defaultdict, deque
from contextvars import ContextVar
import statistics, uuid

# Shared observability objects (module-level singletons)
_log = AgentLogger("agent-service")          # from Lab 4 AgentLogger
_metrics = MetricsCollector()                # from Lab 4 MetricsCollector
_tracer = Tracer()                           # from Lab 4 Tracer
_cache = ResponseCache(ttl_seconds=3600)     # from Lab 3 ResponseCache

# --- Replace the /chat handler body with this instrumented version ---
@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    start = time.time()
    request_id = str(uuid.uuid4())[:8]
    
    _log.info("request_started", request_id=request_id,
              session_id=req.session_id, msg_len=len(req.message))
    
    with _tracer.start_span("chat_request", request_id=request_id) as root_span:
        
        async with _session_lock:
            history = list(sessions.get(req.session_id, [])) if req.session_id else []
            history.append({"role": "user", "content": req.message})
        
        messages = [{"role": "system", "content": "You are a helpful AI engineering assistant."}, *history]
        
        # Check cache — record cache hit as a metric
        cached, was_cached = _cache.get("gpt-4o-mini", messages), False
        if cached:
            answer, was_cached = cached, True
            _log.info("cache_hit", request_id=request_id, session_id=req.session_id)
            _metrics.record_call("gpt-4o-mini", 0, 0, 0, success=True)  # 0 tokens — served from cache
        else:
            with _tracer.start_span("llm_call", model="gpt-4o-mini"):
                llm_start = time.time()
                response = await client.chat.completions.create(model="gpt-4o-mini", messages=messages)
                llm_latency = (time.time() - llm_start) * 1000
            
            answer = response.choices[0].message.content
            usage = response.usage
            
            _cache.set("gpt-4o-mini", messages, answer)
            _metrics.record_call(
                "gpt-4o-mini",
                input_tokens=usage.prompt_tokens,
                output_tokens=usage.completion_tokens,
                latency_ms=llm_latency,
                success=True
            )
            _log.info("llm_call_completed", request_id=request_id,
                      input_tokens=usage.prompt_tokens,
                      output_tokens=usage.completion_tokens,
                      latency_ms=round(llm_latency, 1),
                      cached=False)
    
    history.append({"role": "assistant", "content": answer})
    if req.session_id:
        async with _session_lock:
            sessions[req.session_id] = history[-20:]
            session_last_used[req.session_id] = time.time()
    
    total_latency = (time.time() - start) * 1000
    _log.info("request_completed", request_id=request_id,
              latency_ms=round(total_latency, 1), cached=was_cached)
    
    return ChatResponse(answer=answer, session_id=req.session_id,
                        latency_ms=total_latency, model="gpt-4o-mini")

# --- Add a /metrics endpoint to expose current stats ---
@app.get("/metrics")
async def get_metrics():
    return {
        **_metrics.report(),
        "cache_hit_rate": _cache.hit_rate,
        "active_sessions": len(sessions),
        "active_traces": len(_tracer.spans),
    }
'''

print("Lab 3 + Lab 4 integration pattern:")
print(LAB3_LAB4_INTEGRATION)
print("Key additions to Lab 3:")
print("  1. AgentLogger: every request logs request_id, session_id, latency")
print("  2. MetricsCollector: records latency + tokens per LLM call")
print("  3. Tracer: wraps each request + LLM call in a span")
print("  4. cache_hit_rate: tracked in MetricsCollector, exposed in /metrics")
print("  5. /metrics endpoint: returns current stats snapshot for monitoring")

---
## Summary: Observability Stack for Agents

| Layer | What to use | What it gives you |
|-------|------------|-------------------|
| **Logging** | `python-json-logger` + stdout → CloudWatch/Loki | Searchable, filterable logs |
| **Metrics** | Prometheus + Grafana (or Datadog) | Dashboards, trend analysis |
| **Tracing** | OpenTelemetry → Jaeger OR LangSmith | Full agent trajectory |
| **Alerting** | Grafana Alerts or PagerDuty | Incident response |
| **Evals** | Lab 2's eval suite (run on a schedule) | Quality regression detection |

## The minimum viable observability setup (3 steps)
1. Add `LANGCHAIN_TRACING_V2=true` → instant LangSmith traces
2. Structured JSON logging with `session_id` and `request_id` on every log line
3. One alert on error rate > 5% in any 5-minute window

That covers 80% of production visibility needs in under an hour.

---

## Cross-Lab Notes

### Tracer + asyncio compatibility (X.1)
The `Tracer` in this lab uses `ContextVar` — it is **async-safe** and works correctly with Lab 3's `asyncio.gather`. Each concurrent coroutine gets its own isolated span context. You can use `Tracer` directly inside any `async def` FastAPI handler from Lab 3.

### Unified cost model (X.3)
Labs 2, 3, and 4 each track cost slightly differently:
- **Lab 2** counts cost in `EvalRun.latency_ms` (no per-case cost)
- **Lab 3** tracks cost in `CostTracker` per session
- **Lab 4** tracks cost in `MetricsCollector` per model call

For a unified system, use **Lab 4's `MetricsCollector`** as the single source of truth — wire Lab 3's `CostTracker` to call `metrics.record_call()` on each LLM response, then the `/metrics` endpoint (Part 7 above) reports cost across all labs in one place.

### Cache hit metrics (X.4)
Lab 3's `ResponseCache` tracks `hit_rate`. Lab 4's `/metrics` endpoint in Part 7 above exposes it as `cache_hit_rate`. Add an alert rule for `cache_hit_rate < 0.1` (less than 10% cache efficiency) to catch misconfigured cache keys.

### End-to-end integration (X.5)
After completing all 5 labs, you have all the pieces for a production agent system:
```
User → [Lab 5: Guardrails] → [Lab 3: FastAPI + Sessions]
              ↕                         ↕
       [Lab 1: RAG Pipeline]    [Lab 4: Observability]
              ↕                         ↕
       [Lab 2: Eval Suite] ←→ [CI/CD regression gate]
```
Wire them together in this order:
1. Wrap your Lab 1 RAG agent with Lab 5 guardrails
2. Serve it via Lab 3 FastAPI with session management
3. Instrument with Lab 4 logging + metrics on every `/chat` request
4. Run Lab 2 eval suite on a schedule (daily or pre-deploy) as your quality gate